# Speech Separation Dataset Generation

In [ ]:
import os
import io
import time
import json
import random
import multiprocessing
import numpy as np
import soundfile as sf
import librosa
from collections import defaultdict
from datasets import load_dataset, Audio
from tqdm.auto import tqdm
from joblib import Parallel, delayed

## 1. Configuration

In [ ]:
SEED = 42
SAMPLE_RATE = 16000
MIN_UTTERANCE_SEC = 3.0
MIN_UTTERANCE_SAMPLES = int(SAMPLE_RATE * MIN_UTTERANCE_SEC)

SPEAKER_PROBS = [0.70, 0.20, 0.10]
SPEAKER_COUNTS = [2, 3, 4]

MIN_OVERLAP = 0.30
MAX_OVERLAP = 0.70
MAX_ALLOWED_OVERLAP = 0.75
MIN_START_DELAY = 0.5

GAIN_RANGE_DB = (-4.0, 4.0)
MIN_RMS_DB = -25.0

MAX_RUNTIME_HOURS = 10.0
TIME_SPLIT_RATIOS = {"train": 0.8, "val": 0.1, "test": 0.1}
CHUNK_SIZE = 500

NUM_WORKERS = max(1, multiprocessing.cpu_count() - 1)
OUTPUT_DIR = "./conversational_dataset"
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "checkpoint.json")

np.random.seed(SEED)
print(f"Using {NUM_WORKERS} workers for generation.")

## 2. Dataset Loading

In [ ]:
def load_librispeech_split(split_name):
    ds = load_dataset("openslr/librispeech_asr", "clean", split=split_name)
    ds = ds.cast_column("audio", Audio(decode=False))
    
    speaker_to_indices = defaultdict(list)
    for idx, spk_id in enumerate(ds["speaker_id"]):
        speaker_to_indices[spk_id].append(idx)
        
    print(f"Loaded {split_name}: {len(ds)} utterances from {len(speaker_to_indices)} speakers")
    return ds, speaker_to_indices, list(speaker_to_indices.keys())

train_ds, train_spk_to_idx, train_speakers = load_librispeech_split("train.100")
val_ds, val_spk_to_idx, val_speakers = load_librispeech_split("validation")
test_ds, test_spk_to_idx, test_speakers = load_librispeech_split("test")

## 3. Scheduling & Audio Logic

In [ ]:
def compute_overlap_ratio(start_times, end_times):
    events = []
    for s, e in zip(start_times, end_times):
        events.append((s, 1))
        events.append((e, -1))
    events.sort()
    
    active_speakers = 0
    total_speech_time = 0.0
    overlap_time = 0.0
    last_t = events[0][0]
    
    for t, delta in events:
        dur = t - last_t
        if active_speakers > 0:
            total_speech_time += dur
        if active_speakers > 1:
            overlap_time += dur
            
        active_speakers += delta
        last_t = t
        
    return overlap_time / total_speech_time if total_speech_time > 0 else 0.0

def schedule_speakers(durations, rng):
    num_speakers = len(durations)
    for _ in range(50):
        start_times = [0.0]
        end_times = [durations[0]]
        valid = True
        
        for i in range(1, num_speakers):
            max_start = max(end_times)
            placed = False
            for _ in range(20):
                s = rng.uniform(0, max_start)
                if all(abs(s - st) >= MIN_START_DELAY for st in start_times):
                    start_times.append(s)
                    end_times.append(s + durations[i])
                    placed = True
                    break
            if not placed:
                valid = False
                break
                
        if not valid:
            continue
            
        overlap = compute_overlap_ratio(start_times, end_times)
        if MIN_OVERLAP <= overlap <= MAX_OVERLAP:
            return start_times, overlap
            
    return None, 0.0

def calculate_rms_db(audio):
    rms = np.sqrt(np.mean(audio**2))
    return 20 * np.log10(max(rms, 1e-10))

## 4. Single Sample Generation Pipeline

In [ ]:
def process_sample(args):
    split_name, ds_info, sample_idx, seed = args
    ds, spk_to_idx, available_speakers = ds_info
    
    try:
        srng = np.random.default_rng(seed)
        
        for _ in range(10):
            num_speakers = int(srng.choice(SPEAKER_COUNTS, p=SPEAKER_PROBS))
            speakers = [int(s) for s in srng.choice(available_speakers, size=num_speakers, replace=False)]
            
            signals = []
            utt_ids = []
            durations = []
            gains = []
            
            valid_selection = True
            for spk in speakers:
                indices = spk_to_idx[spk]
                for _ in range(10):
                    utt_idx = srng.choice(indices)
                    item = ds[int(utt_idx)]
                    audio, orig_sr = sf.read(io.BytesIO(item["audio"]["bytes"]))
                    
                    if len(audio) < MIN_UTTERANCE_SAMPLES * (orig_sr / SAMPLE_RATE):
                        continue
                        
                    if orig_sr != SAMPLE_RATE:
                        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=SAMPLE_RATE)
                        
                    rms_db = calculate_rms_db(audio)
                    if rms_db < MIN_RMS_DB:
                        continue
                        
                    gain_db = srng.uniform(*GAIN_RANGE_DB)
                    audio = audio * (10 ** (gain_db / 20))
                    
                    signals.append(audio)
                    durations.append(len(audio) / SAMPLE_RATE)
                    gains.append(gain_db)
                    utt_ids.append(item.get("id", item.get("file", f"{spk}_{utt_idx}")).split("/")[-1].replace(".flac", "").replace(".wav", ""))
                    break
                else:
                    valid_selection = False
                    break
                    
            if not valid_selection:
                continue
                
            start_times, overlap = schedule_speakers(durations, srng)
            if start_times is None or overlap > MAX_ALLOWED_OVERLAP:
                continue
                
            mix_duration = max(s + d for s, d in zip(start_times, durations))
            mix_samples = int(np.ceil(mix_duration * SAMPLE_RATE))
            
            sources = [np.zeros(mix_samples) for _ in range(num_speakers)]
            for i, (audio, start) in enumerate(zip(signals, start_times)):
                start_samp = int(start * SAMPLE_RATE)
                end_samp = start_samp + len(audio)
                sources[i][start_samp:end_samp] = audio
                
            mixture = np.sum(sources, axis=0)
            
            peak = np.max(np.abs(mixture))
            if peak > 0.95:
                scale = 0.95 / peak
                mixture *= scale
                for i in range(num_speakers):
                    sources[i] *= scale
            else:
                scale = 1.0
                
            mix_name = f"sample_{sample_idx:07d}"
            sample_dir = os.path.join(OUTPUT_DIR, split_name, mix_name)
            os.makedirs(sample_dir, exist_ok=True)
            
            sf.write(os.path.join(sample_dir, "mixture.flac"), mixture, SAMPLE_RATE, format='FLAC')
            for i in range(num_speakers):
                sf.write(os.path.join(sample_dir, f"source_{i+1}.flac"), sources[i], SAMPLE_RATE, format='FLAC')
                
            metadata = {
                "mixture_id": mix_name,
                "speaker_ids": speakers,
                "utterance_ids": utt_ids,
                "num_speakers": num_speakers,
                "start_times": [round(float(s), 4) for s in start_times],
                "durations": [round(float(d), 4) for d in durations],
                "gains_db": [round(float(g), 4) for g in gains],
                "overlap_ratio": round(float(overlap), 4),
                "scale_factor": round(float(scale), 6),
                "mix_duration": round(float(mix_duration), 4)
            }
            
            with open(os.path.join(sample_dir, "metadata.json"), "w") as f:
                json.dump(metadata, f, indent=2)
                
            return metadata
            
        return None
        
    except Exception as e:
        return None

## 5. Stateful Execution & Checkpointing

In [ ]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            return json.load(f)
    return {
        "train": {"count": 0, "elapsed_hours": 0.0},
        "val": {"count": 0, "elapsed_hours": 0.0},
        "test": {"count": 0, "elapsed_hours": 0.0},
        "global_seed_offset": 0
    }

def save_checkpoint(state):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(state, f, indent=2)

def generate_split_chunked(split_name, ds_info, allocated_hours, state):
    print(f"\nResuming {split_name} (Allocated: {allocated_hours:.2f}h, Elapsed: {state[split_name]['elapsed_hours']:.2f}h)")
    
    os.makedirs(os.path.join(OUTPUT_DIR, split_name), exist_ok=True)
    start_time = time.time()
    
    while state[split_name]["elapsed_hours"] < allocated_hours:
        current_count = state[split_name]["count"]
        seed_offset = state["global_seed_offset"]
        
        args_list = []
        for i in range(CHUNK_SIZE):
            idx = current_count + i
            args_list.append((split_name, ds_info, idx, SEED + seed_offset + i))
            
        results = Parallel(n_jobs=NUM_WORKERS, backend="loky")(
            delayed(process_sample)(arg) for arg in tqdm(args_list, desc=f"{split_name} (Chunk)")
        )
        
        successful = sum(1 for res in results if res is not None)
                    
        elapsed_chunk_hours = (time.time() - start_time) / 3600.0
        
        state[split_name]["count"] += successful
        state[split_name]["elapsed_hours"] += elapsed_chunk_hours
        state["global_seed_offset"] += CHUNK_SIZE
        save_checkpoint(state)
        
        start_time = time.time() 
        print(f"  -> Generated {successful} samples. Split total: {state[split_name]['count']}. Time left: {allocated_hours - state[split_name]['elapsed_hours']:.2f}h")
        
    print(f"Finished {split_name} time allocation.")

## 6. Run Time-Bounded Generation

In [ ]:

state = load_checkpoint()
    
splits = [
    ("train", (train_ds, train_spk_to_idx, train_speakers), TIME_SPLIT_RATIOS["train"] * MAX_RUNTIME_HOURS),
    ("val", (val_ds, val_spk_to_idx, val_speakers), TIME_SPLIT_RATIOS["val"] * MAX_RUNTIME_HOURS),
    ("test", (test_ds, test_spk_to_idx, test_speakers), TIME_SPLIT_RATIOS["test"] * MAX_RUNTIME_HOURS)
]
    
for split_name, ds_info, allocated_hours in splits:
    if state[split_name]["elapsed_hours"] < allocated_hours:
            generate_split_chunked(split_name, ds_info, allocated_hours, state)
    else:
        print(f"\n{split_name} already completed its allocated time ({allocated_hours:.2f}h).")
            
print("\n\nAll time allocations completed!")